# HDS SDOH Data Validation Tests

In [ ]:
workspace_id = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""
deployment_environment = ""
silver_lakehouse_id=""
bronze_lakehouse_id=""


In [ ]:
from pyspark.sql.functions import input_file_name, col, count, countDistinct, coalesce, lit
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
from concurrent.futures import ThreadPoolExecutor
import unittest
from pyspark.sql import SparkSession
import io
import logging
import sempy.fabric as fabric
import xmlrunner

class SDOHDataValidationTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, workspace_id = None, bronze_lakehouse_id = None, databases = []):
        super().__init__(methodName)
        logging.basicConfig()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id
        self.databases = databases
        self.sdoh_table_prefix='Sdoh'
        self.silver_lakehouse_id=silver_lakehouse_id

    def test_sdoh_bronze_ingestion_folder_is_empty(self):
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/SDOH/CSV"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/SDOH/XLS"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/SDOH/XLSX"))
        ingest_files_folder_csv = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/SDOH/CSV")
        ingest_files_folder_xls = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/SDOH/XLS")
        ingest_files_folder_xlsx = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/SDOH/XLSX")
        
        # Assert the sample data folder exists but is empty
        self.assertEqual(1, len(ingest_files_folder_csv))
        sample_data_folder_csv = mssparkutils.fs.ls(ingest_files_folder_csv[0].path)
        self.assertEqual(0, len(sample_data_folder_csv))
        self.assertEqual(0, len(ingest_files_folder_xls))
        self.assertEqual(6, len(ingest_files_folder_xlsx))
        total_xlsx_files_count = check_for_xlsx_files(ingest_files_folder_xlsx)
        self.assertEqual(0, total_xlsx_files_count) 

    def test_bronze_lakehouse_folder_structure_is_hydrated(self):
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/External"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Failed"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Process"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/ReferenceData"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/SampleData"))

    def test_bronze_ingestion_processed_sample_data(self):

        processed_files_folder = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Process/SDOH/CSV/")
        self.assertEqual(1, len(processed_files_folder))

        year_folders = mssparkutils.fs.ls(processed_files_folder[0].path)
        self.assertEqual(1, len(year_folders))

        months_folder = mssparkutils.fs.ls(year_folders[0].path)
        self.assertEqual(1, len(months_folder))

        days_folder = mssparkutils.fs.ls(months_folder[0].path)
        self.assertEqual(1, len(days_folder))

        dataset_folder = mssparkutils.fs.ls(days_folder[0].path)
        self.assertEqual(1, len(dataset_folder))
        
        sample_data_folder = mssparkutils.fs.ls(dataset_folder[0].path)
        self.assertEqual(4, len(sample_data_folder))
        data_folder_xlsx=f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Process/SDOH/XLSX"
        total_file_processed=check_for_xlsx_files(data_folder_xlsx)
        self.assertEqual(7, total_file_processed)
    
    def test_sdoh_reference_data_folder_contains_ziptofipsmapping_file(self):
        sdoh_reference_data_folder = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/ReferenceData/SDOH")
        self.assertEqual(1, len(sdoh_reference_data_folder))

        location_dataset = mssparkutils.fs.ls(sdoh_reference_data_folder[0].path)
        self.assertEqual(1, len(location_dataset))

    def test_sdoh_tables_contains_metadata_column_and_sdoh__bronze_tables_contains_data(self):
        tables = self.spark.sql(f"SHOW TABLES IN `{self.bronze_lakehouse_id}`").collect()
        processed_tables = [table for table in tables if self.sdoh_table_prefix.lower() in table.tableName.lower()]
        tables_with_datasetid=self.filter_tables_with_column("DataSetMetadataId",processed_tables)
        self.assertEqual(len(tables_with_datasetid), len(processed_tables))
        for table in tables_with_datasetid:
            df_1 = self.spark.sql(f"SELECT COUNT(*) AS count FROM {table.tableName}")
            record_count = df_1.collect()[0]["count"]     
            self.assertGreater(record_count, 0, "Each SDOH table should have non-zero counts.")
    
    def test_sdoh_datasetmetadata_table_rows_count(self):
        df_dataset_meta_data = spark.sql(f"SELECT  COUNT(*) AS count FROM SdohDataSetMetadata")
        record_count = df_dataset_meta_data.collect()[0]["count"]
        self.assertEqual(8, record_count)

    def test_sdoh_silver_table_contains_data(self):
        table_names = ['SocialDeterminant','SocialDeterminantCategory','SocialDeterminantDataSetMetadata','SocialDeterminantSubCategory','UnitOfMeasure']
        for table_name in table_names:
            count_df = spark.sql(f"SELECT COUNT(*) AS count FROM `{self.silver_lakehouse_id}`.{table_name}")
            count = count_df.collect()[0]["count"]
            self.assertGreater(count, 0, "Each SDOH table should have non-zero counts.")

    def test_sdoh_compare_source_target_data_social_determinant(self):
        source_query = f"""SELECT COUNT(*) as count FROM SdohLayout l INNER JOIN (SELECT s.SocialDeterminantCode,CONCAT(county, fips, state) AS row_id FROM 
            ( SELECT county, fips,state,STACK(
                        13, PCTPOV017, 'PCTPOV017',POVALL, 'POVALL', 
                        Num_inPOV_0_17_ACS, 'Num_inPOV_0_17_ACS', Poverty_Rate_0_17_ACS, 'Poverty_Rate_0_17_ACS', 
                        PerCapitaInc, 'PerCapitaInc', POV017, 'POV017',  Deep_Pov_Children, 'Deep_Pov_Children', PCTPOVALL,
                       'PCTPOVALL', Poverty_Rate_ACS, 'Poverty_Rate_ACS',NumAll_inPOV_ACS, 'NumAll_inPOV_ACS',Median_HH_Inc_ACS,
                       'Median_HH_Inc_ACS',Deep_Pov_All, 'Deep_Pov_All', MedHHInc, 'MedHHInc'
                    ) AS (SocialDeterminantValue, SocialDeterminantCode) FROM SdohRuralAtlasIncome) s 
                    LATERAL VIEW EXPLODE(ARRAY(FIPS)) exploded_table AS LocationType ) s1 
                    ON l.SocialDeterminantName = s1.SocialDeterminantCode INNER JOIN SdohDataSetMetadata m ON l.DataSetMetadataId = m.DataSetMetadataId """
        
        target_query=f""" SELECT count(*) as count from `{self.silver_lakehouse_id}`.SocialDeterminant where SourceTable='SdohRuralAtlasIncome_SocialDeterminant_SDOH' """
        source_df=spark.sql(source_query)
        target_df=spark.sql(target_query)
        source_count= source_df.collect()[0]["count"]
        target_count= target_df.collect()[0]["count"]
        self.assertEqual(source_count, target_count)

    def has_column(self,table_name, col_name):
        df = self.spark.table(table_name)
        normalized_columns = [col.strip().lower() for col in df.columns]
        return col_name.strip().lower() in normalized_columns

    def filter_tables_with_column(self,column_name,processed_tables):
        tables_with_column = [table for table in processed_tables if self.has_column(f"`{self.bronze_lakehouse_id}`.{table.tableName}", column_name)]
        return tables_with_column if tables_with_column else []


def check_for_xlsx_files(directory_path):
    xlsx_files_count = 0
    try:
        folder_contents = mssparkutils.fs.ls(directory_path)
        for item in folder_contents:
            if item.name.endswith(".xlsx"):
                xlsx_files_count += 1
            elif item.isDir:  # If the item is a directory, recursively check its contents
                xlsx_files_count += check_for_xlsx_files(item.path)
    except Exception as e:
        print(f"Error: {e}")
    return xlsx_files_count



def run_tests_and_write_output(spark):
    
    # Load and run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(SDOHDataValidationTests)

    # Inject parameters / context to the tests
    for test in suite:
        test.spark = spark
        test.workspace_id = workspace_id
        test.bronze_lakehouse_id=bronze_lakehouse_id
        test.silver_lakehouse_id=silver_lakehouse_id
    # Write XML test report to stream, decode after completion
    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream, verbosity=3).run(suite)
    xml_output = write_stream.getvalue().decode('utf-8')

    # Write report to lakehouse
    mssparkutils.fs.put(f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)
    return xml_output

report = run_tests_and_write_output(spark)